# 🏭 Analiz 5: Garaj Uzmanlık ve Tamir Kalitesi — Tam Kanıt Zinciri

**Hedef:** Hangi garaj iyi tamir ediyor? Garaj × Marka × Sistem uyumu var mı? Tamir kalitesi cascade pattern'iyle ölçülebilir mi?

**Metodoloji notu:**
- ariza_model.csv (sadece İETT öz filosu — ÖHO/KOOP yok)
- Bakım türü (değişim vs tamir) verisi YOK — sebep yorumu sınırlı
- Analiz 1, 2, 3, 4'ten dersler uygulandı (leakage testi, confounder kontrolü)
- ML kararı VERİYE göre verilir (şartlama yok)

---

## 1. Veri + Garaj Profil Tablosu

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# Veri yukleme
df = pd.read_csv('../panel_data/temiz_veri/ariza_model.csv')
df['OLAYTARIHI'] = pd.to_datetime(df['OLAYTARIHI'], format='mixed')
df = df.sort_values(['KAPINO', 'OLAYTARIHI']).reset_index(drop=True)

# Sefer verisi
arac_hatlar = pd.read_csv('../panel_data/temiz_veri/arac_gunluk_hatlar.csv')
SEFER_KOL = [c for c in arac_hatlar.columns if 'SEFER' in c.upper()][0]
arac_sefer = arac_hatlar.groupby('KAPINO')[SEFER_KOL].sum().reset_index()
arac_sefer.columns = ['KAPINO', 'toplam_sefer']

print('=== TEMEL ISTATISTIK ===')
print(f'Toplam ariza: {len(df):,}')
print(f'Benzersiz arac: {df["KAPINO"].nunique():,}')
print(f'Benzersiz garaj: {df["GARAJ"].nunique()}')
print(f'Benzersiz marka: {df["MARKA"].nunique()}')

# Garaj profil tablosu
arac_garaj = df.groupby('KAPINO')['GARAJ'].agg(lambda x: x.mode().iloc[0]).reset_index()
arac_yas  = df.groupby('KAPINO')['MODELYILI'].agg(lambda x: x.mode().iloc[0]).reset_index()
arac_yas['yas'] = 2025 - arac_yas['MODELYILI']
arac_marka = df.groupby('KAPINO')['MARKA'].agg(lambda x: x.mode().iloc[0]).reset_index()

arac_meta = arac_garaj.merge(arac_yas[['KAPINO','yas']], on='KAPINO')
arac_meta = arac_meta.merge(arac_marka, on='KAPINO')
arac_meta = arac_meta.merge(arac_sefer, on='KAPINO', how='left')

# Garaj seviyesinde aggrege
garaj_profil = arac_meta.groupby('GARAJ').agg(
    n_arac=('KAPINO', 'nunique'),
    ort_yas=('yas', 'mean'),
    med_yas=('yas', 'median'),
    ort_sefer=('toplam_sefer', 'mean'),
    toplam_sefer=('toplam_sefer', 'sum'),
).round(1).reset_index()

# Garaj başına ariza istatistikleri
garaj_ariza = df.groupby('GARAJ').agg(
    toplam_ariza=('KAPINO', 'count'),
    ort_skor=('ciddiyet_skoru', 'mean'),
    ciddi_oran=('ciddi_ariza', 'mean'),
).round(3).reset_index()

garaj_profil = garaj_profil.merge(garaj_ariza, on='GARAJ')
garaj_profil['ariza_per_1000_sefer'] = (garaj_profil['toplam_ariza'] / garaj_profil['toplam_sefer'] * 1000).round(2)

print()
print('=== GARAJ PROFIL TABLOSU ===')
print(garaj_profil.sort_values('toplam_ariza', ascending=False).to_string(index=False))

# Kayitsiz Arac garajini cikar (ozellikle 25 kayit)
garaj_profil_clean = garaj_profil[garaj_profil['GARAJ'] != 'Kayıtsız Araç'].copy()
print()
print(f'NOT: "Kayitsiz Arac" cikarildi (sadece 25 kayit, 1 arac)')
print(f'Analiz seti: {len(garaj_profil_clean)} garaj')

# Onceden bilinenler (Analiz 3 Hucre F)
print()
print('=== ONCEDEN BILINENLER (Analiz 3 Hucre F regresyon) ===')
print('Topkapi:    en genc filo  (-14.96 yil Anadolu refrans)')
print('Hasanpasa:  -10.43 yil')
print('IKITELLI:   -8.05 yil')
print('Sahinkaya:  +0.36 yil (Anadolu seviyesinde yasli)')
print('Anadolu en yasli filo + zor hatlara (eğim) hizmet veriyor.')


=== TEMEL ISTATISTIK ===
Toplam ariza: 58,559
Benzersiz arac: 3,509
Benzersiz garaj: 12
Benzersiz marka: 6

=== GARAJ PROFIL TABLOSU ===
                    GARAJ  n_arac  ort_yas  med_yas  ort_sefer  toplam_sefer  toplam_ariza  ort_skor  ciddi_oran  ariza_per_1000_sefer
                Hasanpaşa     319      8.0     10.0     1525.7        486707          9053     3.746       0.421                 18.60
IKITELLIISLETTIRMEGARAJI2     330     11.3     12.0     1036.1        341899          8536     3.496       0.384                 24.97
               Edirnekapı     381     11.8     16.0     1344.7        512326          7818     3.713       0.403                 15.26
                  KURTKÖY     346     12.1     12.0     1199.1        414885          7265     3.860       0.406                 17.51
         SULTANGAZIGARAJI     413     12.2     12.0     1251.7        516949          7252     3.437       0.388                 14.03
                  Anadolu     353     17.4     19.0  

---

## 2. Garaj × Marka Uyumu Matrisi
Hangi garaj hangi markada anormal yüksek ciddi arıza üretiyor? Chi² + Lift skoru.

In [2]:
# BOLUM 2: Garaj × Marka Uyumu Matrisi
# Soru: Hangi garaj hangi markayi iyi/kotu performans gosteriyor?
# Lift = P(ciddi|garaj×marka) / P(ciddi|marka_genel)

# Filtre: en cok arıza yapan 5 marka
top_markalar = df['MARKA'].value_counts().head(5).index.tolist()
df_clean = df[(df['GARAJ'] != 'Kayıtsız Araç') & (df['MARKA'].isin(top_markalar))].copy()

print(f'Top 5 marka: {top_markalar}')
print(f'Analiz seti: {len(df_clean):,} ariza')

# Marka genel ciddi_oran
marka_genel = df_clean.groupby('MARKA')['ciddi_ariza'].mean().to_dict()
print()
print('=== GENEL MARKA CIDDI ORANI ===')
for m, p in marka_genel.items():
    print(f'  {m}: {p*100:.1f}%')

# Garaj × Marka matrisi
matrix = df_clean.groupby(['GARAJ', 'MARKA']).agg(
    n=('ciddi_ariza', 'count'),
    ciddi=('ciddi_ariza', 'mean'),
    skor=('ciddiyet_skoru', 'mean'),
).reset_index()

# Min 30 kayit (gurultu eleme)
matrix = matrix[matrix['n'] >= 30].copy()
matrix['marka_genel'] = matrix['MARKA'].map(marka_genel)
matrix['ciddi_lift'] = matrix['ciddi'] / matrix['marka_genel']

print()
print('=== GARAJ × MARKA — EN YUKSEK LIFT (kotu performans) ===')
top_kotu = matrix.nlargest(15, 'ciddi_lift')[['GARAJ','MARKA','n','ciddi','marka_genel','ciddi_lift']].round(3)
print(top_kotu.to_string(index=False))

print()
print('=== GARAJ × MARKA — EN DUSUK LIFT (iyi performans) ===')
top_iyi = matrix.nsmallest(10, 'ciddi_lift')[['GARAJ','MARKA','n','ciddi','marka_genel','ciddi_lift']].round(3)
print(top_iyi.to_string(index=False))

# Chi-square test: garaj × marka × ciddiyet bagimli mi?
print()
print('=== CHI-SQUARE: GARAJ × MARKA × CIDDIYET ===')
for marka in top_markalar:
    sub = df_clean[df_clean['MARKA'] == marka]
    if sub['GARAJ'].nunique() < 3: continue
    ct = pd.crosstab(sub['GARAJ'], sub['ciddi_ariza'])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    print(f'{marka:12s}: chi²={chi2:.1f}, df={dof}, p={p:.6f}', '←ANLAMLI' if p<0.05 else '')

# Heatmap data
print()
print('=== HEATMAP: Garaj × Marka Ciddi Lift ===')
pivot = matrix.pivot_table(index='GARAJ', columns='MARKA', values='ciddi_lift')
print(pivot.round(2).to_string())


Top 5 marka: ['MERCEDES', 'OTOKAR', 'KARSAN', 'BMC', 'AKIA']
Analiz seti: 56,028 ariza

=== GENEL MARKA CIDDI ORANI ===
  AKIA: 32.9%
  BMC: 36.1%
  KARSAN: 41.1%
  MERCEDES: 38.7%
  OTOKAR: 38.7%

=== GARAJ × MARKA — EN YUKSEK LIFT (kotu performans) ===
                    GARAJ    MARKA    n  ciddi  marka_genel  ciddi_lift
IKITELLIISLETTIRMEGARAJI2      BMC 1116  0.469        0.361       1.298
               Edirnekapı     AKIA 1771  0.420        0.329       1.279
                Hasanpaşa   OTOKAR 2924  0.481        0.387       1.242
           IKITELLIGARAJI MERCEDES 1151  0.442        0.387       1.143
                Şahinkaya MERCEDES 1136  0.426        0.387       1.101
         SULTANGAZIGARAJI MERCEDES 1807  0.422        0.387       1.089
IKITELLIISLETTIRMEGARAJI2   KARSAN 4713  0.444        0.411       1.079
               Edirnekapı MERCEDES 6047  0.399        0.387       1.030
                  KURTKÖY   OTOKAR 4351  0.397        0.387       1.027
                  KURTKÖY

---

## 3. Tamir Kalitesi Metriği (Garaj Cascade Oranı)
Garaj başına 24 saat içinde tekrar arıza oranı. Yüksek = tamir sonrası tekrar arıza (bakım kalitesi şüphesi).

In [3]:
# BOLUM 3: Tamir Kalitesi Metriği — Garaj Cascade Oranı
# Soru: Hangi garaj tamir sonrasi tekrar ariza yaratiyor?
# Yontem: Garaj başına cascade (24s tekrar ariza) orani

# Cascade tanimi: ardisik ariza 24s icinde
df['onceki_t'] = df.groupby('KAPINO')['OLAYTARIHI'].shift(1)
df['saat_fark'] = (df['OLAYTARIHI'] - df['onceki_t']).dt.total_seconds() / 3600
df['cascade_24s'] = (df['saat_fark'] < 24).fillna(False)

# 7g cascade
df['cascade_7g'] = (df['saat_fark'] < 168).fillna(False)

# Garaj başına cascade orani
garaj_cas = df[df['GARAJ'] != 'Kayıtsız Araç'].groupby('GARAJ').agg(
    n_ariza=('cascade_24s', 'count'),
    cascade_24s_n=('cascade_24s', 'sum'),
    cascade_24s_oran=('cascade_24s', 'mean'),
    cascade_7g_n=('cascade_7g', 'sum'),
    cascade_7g_oran=('cascade_7g', 'mean'),
).round(3).reset_index()

# Filo bazinda toplam ortalama (kıyas için)
toplam_cas24 = df['cascade_24s'].mean()
toplam_cas7g = df['cascade_7g'].mean()
print(f'Tum filo cascade_24s orani: {toplam_cas24:.3f}')
print(f'Tum filo cascade_7g orani:  {toplam_cas7g:.3f}')

# Garaj lift (cascade)
garaj_cas['cas24s_lift'] = garaj_cas['cascade_24s_oran'] / toplam_cas24
garaj_cas['cas7g_lift']  = garaj_cas['cascade_7g_oran'] / toplam_cas7g

print()
print('=== GARAJ BAZLI TAMIR KALITESI (cascade=tekrar ariza) ===')
print('Yuksek lift = tamirden sonra ariza yine cikıyor = bakım kalitesi şüpheli')
print()
print(garaj_cas.sort_values('cas24s_lift', ascending=False).to_string(index=False))

# Chi-square: garaj × cascade bagimli mi?
print()
print('=== CHI-SQUARE: GARAJ × CASCADE_24S ===')
df_c = df[df['GARAJ'] != 'Kayıtsız Araç'].copy()
ct = pd.crosstab(df_c['GARAJ'], df_c['cascade_24s'])
chi2, p, dof, _ = stats.chi2_contingency(ct)
print(f'chi²={chi2:.1f}, df={dof}, p={p:.6f}')
if p < 0.05:
    print('VERIYE GORE: Garaj × cascade ANLAMLI bagli — garaj kalitesi farkliligi var.')
else:
    print('VERIYE GORE: Garaj × cascade anlamli farkliligi YOK.')

# Yorum
print()
print('=== ONEMLI NOT ===')
print('Cascade orani "kotu tamir" demek DEGIL kesin olarak — bakim turu (degisim/tamir) verimiz yok.')
print('Yuksek cascade orani: tamir kalitesi sorunu OLABILIR, ya da o garaj zaten kronik sorunlu')
print('  araclara hizmet ediyor olabilir. Confounder kontrolu Bolum 6\'da.')


Tum filo cascade_24s orani: 0.149
Tum filo cascade_7g orani:  0.557

=== GARAJ BAZLI TAMIR KALITESI (cascade=tekrar ariza) ===
Yuksek lift = tamirden sonra ariza yine cikıyor = bakım kalitesi şüpheli

                    GARAJ  n_ariza  cascade_24s_n  cascade_24s_oran  cascade_7g_n  cascade_7g_oran  cas24s_lift  cas7g_lift
                Hasanpaşa     9053           1817             0.201          6266            0.692     1.348729    1.242460
IKITELLIISLETTIRMEGARAJI2     8536           1576             0.185          5656            0.663     1.241368    1.190391
               Edirnekapı     7818           1265             0.162          4651            0.595     1.087035    1.068300
                  KURTKÖY     7265           1083             0.149          4442            0.611     0.999804    1.097027
         SULTANGAZIGARAJI     7252           1001             0.138          3769            0.520     0.925993    0.933640
                  Anadolu     4900            590      

---

## 4. Garaj × Sistem Uzmanlığı
Hangi garaj hangi sistemde dominant? Uzmanlık mı yoksa kronik sorun mu?

In [4]:
# BOLUM 4: Garaj × Sistem Uzmanligi
# Soru: Hangi garaj hangi sistemde dominant?
# Bu uzmanlik mi (iyi tamir) yoksa kronik sorun mu (kotu tamir)?

df_c = df[df['GARAJ'] != 'Kayıtsız Araç'].copy()

# Garaj × Sistem matrisi
matrix_s = df_c.groupby(['GARAJ', 'ARIZAUSTKODTANIM']).agg(
    n=('ciddi_ariza', 'count'),
    ciddi=('ciddi_ariza', 'mean'),
    skor=('ciddiyet_skoru', 'mean'),
).reset_index()

# Min 50 kayit
matrix_s = matrix_s[matrix_s['n'] >= 50].copy()

# Sistem genel orani
sistem_genel = df_c.groupby('ARIZAUSTKODTANIM').agg(
    n_total=('ciddi_ariza','count'),
    ciddi_genel=('ciddi_ariza','mean'),
).reset_index()
sistem_genel = sistem_genel[sistem_genel['n_total'] >= 200]  # min 200 toplam kayit

matrix_s = matrix_s.merge(sistem_genel[['ARIZAUSTKODTANIM','ciddi_genel']], on='ARIZAUSTKODTANIM')
matrix_s['lift'] = matrix_s['ciddi'] / matrix_s['ciddi_genel']

# En yuksek lift = bu garajda bu sistem kotu
print('=== GARAJ × SISTEM — EN YUKSEK CIDDI LIFT (top 20) ===')
print('Yorum: Bu sistem bu garajda diger garajlardan daha kotu performans')
print()
top_kotu = matrix_s.nlargest(20, 'lift')[['GARAJ','ARIZAUSTKODTANIM','n','ciddi','ciddi_genel','lift']].round(3)
print(top_kotu.to_string(index=False))

# En dusuk lift = bu garajda bu sistem uzman
print()
print('=== GARAJ × SISTEM — EN DUSUK CIDDI LIFT (uzmanlik isareti, top 10) ===')
print('Yorum: Bu sistem bu garajda diger garajlardan daha iyi performans')
print()
top_iyi = matrix_s.nsmallest(15, 'lift')[['GARAJ','ARIZAUSTKODTANIM','n','ciddi','ciddi_genel','lift']].round(3)
print(top_iyi.to_string(index=False))

# Garaj bazli "uzmanlik" skoru: bu garajda en az lift kac sistemde
print()
print('=== GARAJ "UZMANLIK SKORU" (lift<0.85 olan sistem sayisi) ===')
uzmanlik = matrix_s[matrix_s['lift'] < 0.85].groupby('GARAJ').size().reset_index(name='uzman_sistem_n')
print(uzmanlik.sort_values('uzman_sistem_n', ascending=False).to_string(index=False))

# Sorunlu garaj: lift>1.15 olan sistem sayisi
print()
print('=== GARAJ "SORUN SKORU" (lift>1.15 olan sistem sayisi) ===')
sorun = matrix_s[matrix_s['lift'] > 1.15].groupby('GARAJ').size().reset_index(name='sorunlu_sistem_n')
print(sorun.sort_values('sorunlu_sistem_n', ascending=False).to_string(index=False))


=== GARAJ × SISTEM — EN YUKSEK CIDDI LIFT (top 20) ===
Yorum: Bu sistem bu garajda diger garajlardan daha kotu performans

           GARAJ                 ARIZAUSTKODTANIM    n  ciddi  ciddi_genel  lift
         Topkapı         KAMERA SİSTEMİ ARIZALARI  162  0.099        0.028 3.471
       Hasanpaşa    YAKIT ve ENJEKSİYON ARIZALARI   79  0.785        0.357 2.196
      Edirnekapı    YAKIT ve ENJEKSİYON ARIZALARI   69  0.667        0.357 1.866
      Edirnekapı           KAYIŞ KASNAK ARIZALARI  207  0.372        0.202 1.844
         Topkapı                  AKBİL ARIZALARI  191  0.037        0.020 1.843
      Edirnekapı      BASINÇLI MOTOR YAĞI SİSTEMİ  122  0.270        0.150 1.801
       Hasanpaşa           KAYIŞ KASNAK ARIZALARI   87  0.356        0.202 1.766
       Hasanpaşa      BASINÇLI HAVA HATTI ARIZASI  110  0.564        0.343 1.643
       Kağıthane                  AKBİL ARIZALARI   62  0.032        0.020 1.623
        Sarıgazi                KAROSER ARIZALARI  133  0.444      

---

## 5. Random Null Test — Garaj Etkisi Gerçek mi?
Garaj varyansı rastgele permutasyondan anlamlı yüksek mi? (Analiz 2 metodolojisi)

In [5]:
# BOLUM 5: Random Null Test — Garaj Etkisi Gercek mi?
# Soru: Garaj ciddi_oran dagilimi sansa bagli olabilir mi?
# Yontem: Garaj atamalarini rastgele permute et, gercek varyansla karsilastir

garaj_filt = df[df['GARAJ'] != 'Kayıtsız Araç'].copy()
garaj_gercek = garaj_filt.groupby('GARAJ')['ciddiyet_skoru'].mean()
gercek_var = garaj_gercek.var()

np.random.seed(42)
permute_var = []
for _ in range(100):
    df_p = garaj_filt.copy()
    df_p['GARAJ'] = np.random.permutation(df_p['GARAJ'].values)
    var_p = df_p.groupby('GARAJ')['ciddiyet_skoru'].mean().var()
    permute_var.append(var_p)

permute_var = np.array(permute_var)
print('=== GARAJ ETKISI NULL TEST (100 PERMUTASYON) ===')
print(f'Gercek garaj ort_skor varyansi: {gercek_var:.4f}')
print(f'Permute median varyans:          {np.median(permute_var):.4f}')
print(f'Permute P95 varyans:             {np.percentile(permute_var, 95):.4f}')
print(f'Gercek varyans permute %X yuksek: {(gercek_var > permute_var).mean()*100:.0f}%')

if gercek_var > np.percentile(permute_var, 95):
    print('VERIYE GORE: Garaj varyansi random permutasyondan ANLAMLI YUKSEK.')
    print('  -> Garajlar arasi ciddiyet farki GERCEK.')
else:
    print('VERIYE GORE: Garaj varyansi random ile uyumlu.')
    print('  -> Garaj etkisi sansa bagli olabilir.')

# Ayni test cascade_24s icin
print()
print('=== GARAJ ETKISI — CASCADE_24S NULL TEST ===')
gercek_cas = garaj_filt.groupby('GARAJ')['cascade_24s'].mean().var()
permute_cas = []
np.random.seed(42)
for _ in range(100):
    df_p = garaj_filt.copy()
    df_p['GARAJ'] = np.random.permutation(df_p['GARAJ'].values)
    permute_cas.append(df_p.groupby('GARAJ')['cascade_24s'].mean().var())
permute_cas = np.array(permute_cas)
print(f'Gercek cascade varyansi: {gercek_cas:.5f}')
print(f'Permute P95: {np.percentile(permute_cas, 95):.5f}')
if gercek_cas > np.percentile(permute_cas, 95):
    print('VERIYE GORE: Garaj-cascade iliskisi ANLAMLI gercek.')
else:
    print('VERIYE GORE: Garaj-cascade iliskisi random ile uyumlu.')


=== GARAJ ETKISI NULL TEST (100 PERMUTASYON) ===
Gercek garaj ort_skor varyansi: 0.2401
Permute median varyans:          0.0011
Permute P95 varyans:             0.0033
Gercek varyans permute %X yuksek: 100%
VERIYE GORE: Garaj varyansi random permutasyondan ANLAMLI YUKSEK.
  -> Garajlar arasi ciddiyet farki GERCEK.

=== GARAJ ETKISI — CASCADE_24S NULL TEST ===
Gercek cascade varyansi: 0.00158
Permute P95: 0.00009
VERIYE GORE: Garaj-cascade iliskisi ANLAMLI gercek.


---

## 6. Confounder Kontrolü (Multiple Regression)
Yaş + sefer + hat eğimi kontrol altında garaj etkisi kalıyor mu? (Analiz 3 cross-ref)

In [6]:
# BOLUM 6: Confounder Kontrolu — Yaş + Sefer + Hat Zorluk
# Soru: Garaj ciddi_oran farki gercekten garaj kalitesinden mi yoksa
# yaş/sefer/hat farkindan mi geliyor?
# Yontem: Multiple regression M1→M4

import statsmodels.api as sm
from statsmodels.formula.api import ols

# Arac bazinda profil
arac_full = arac_meta.merge(
    df.groupby('KAPINO').agg(
        ariza_n=('ciddi_ariza','count'),
        ort_skor=('ciddiyet_skoru','mean'),
        ciddi_oran=('ciddi_ariza','mean'),
        cascade_oran=('cascade_24s','mean'),
    ).reset_index(),
    on='KAPINO', how='inner'
)

# Hat zorluk (Analiz 3'ten egim_puan)
# arac_gunluk_hatlar × hat_elevation join
import json
with open(r'panel_data\hat_elevation.json', encoding='utf-8') as f:
    hat_elev_raw = json.load(f)

def mm_norm(s, q=None):
    if q is not None:
        s = s.clip(upper=s.quantile(q))
    mn, mx = s.min(), s.max()
    return ((s - mn) / (mx - mn) * 100).round(1) if mx > mn else pd.Series(0.0, index=s.index)

hat_elev = pd.DataFrame([
    {'HATKODU': k, 'rakim_fark': v.get('rakım_farkı', 0), 'tirmanma_m': v.get('tırmanma_m', 0)}
    for k, v in hat_elev_raw.items()
])
hat_elev['norm_rakim'] = mm_norm(hat_elev['rakim_fark'], q=0.99)
hat_elev['norm_tirm'] = mm_norm(hat_elev['tirmanma_m'], q=0.99)
hat_elev['egim_puan'] = (hat_elev['norm_rakim'] * 0.4 + hat_elev['norm_tirm'] * 0.6).round(1)

# Arac başına agirlikli egim
ah = arac_hatlar.merge(hat_elev[['HATKODU','egim_puan']], on='HATKODU', how='left')
arac_egim = ah.dropna(subset=['egim_puan']).groupby('KAPINO').apply(
    lambda g: np.average(g['egim_puan'], weights=g[SEFER_KOL].clip(lower=0.01))
).reset_index(name='ort_egim')

arac_full = arac_full.merge(arac_egim, on='KAPINO', how='left')
arac_full = arac_full[arac_full['GARAJ'] != 'Kayıtsız Araç'].copy()
arac_full['log_sefer'] = np.log1p(arac_full['toplam_sefer'].fillna(0))
arac_full = arac_full.dropna(subset=['yas','log_sefer','ort_egim','ort_skor','GARAJ'])
print(f'Regresyon seti: {len(arac_full):,} arac')

# Modeller — bagimli degisken: ort_skor
m1 = ols('ort_skor ~ C(GARAJ)', data=arac_full).fit()
m2 = ols('ort_skor ~ C(GARAJ) + yas', data=arac_full).fit()
m3 = ols('ort_skor ~ C(GARAJ) + yas + log_sefer', data=arac_full).fit()
m4 = ols('ort_skor ~ C(GARAJ) + yas + log_sefer + ort_egim', data=arac_full).fit()

print()
print('=== R² karsilastirma ===')
print(f'M1 (sadece garaj):                R²={m1.rsquared:.3f}')
print(f'M2 (+yas):                          R²={m2.rsquared:.3f}')
print(f'M3 (+yas+sefer):                    R²={m3.rsquared:.3f}')
print(f'M4 (+yas+sefer+egim):               R²={m4.rsquared:.3f}')

# Garaj sabit etkilerini M4'ten oku (Anadolu referans)
print()
print('=== M4: GARAJ SABIT ETKILERI (referans = ilk garaj alfabetik) ===')
print('Negatif = referans garaj daha kotu, pozitif = referans daha iyi')
print()
for name, val in sorted(m4.params.items()):
    if 'GARAJ' in name:
        p_val = m4.pvalues[name]
        isaret = '✓' if p_val < 0.05 else ' '
        print(f'{isaret} {name[:50]:50s}  k={val:+.4f}  p={p_val:.4f}')

print()
print('=== YORUM ===')
print('R² M1→M4 farki: garaj etkisi degisiyor mu?')
fark_m4 = m4.rsquared - m1.rsquared
if fark_m4 > 0.05:
    print(f'  Confounder ekleyince R² {fark_m4:+.3f} arti → diger faktorler ek aciklama saglıyor.')
if m4.rsquared > m1.rsquared * 0.7:
    print('  Garaj etkisi confounder altinda ÇOĞUNLUKLA KORUNUYOR.')
else:
    print('  Garaj etkisi confounder altinda BUYUK OLCUDE AZALDI — bagimsiz garaj etkisi zayif.')


Regresyon seti: 3,509 arac

=== R² karsilastirma ===
M1 (sadece garaj):                R²=0.236
M2 (+yas):                          R²=0.237
M3 (+yas+sefer):                    R²=0.256
M4 (+yas+sefer+egim):               R²=0.256

=== M4: GARAJ SABIT ETKILERI (referans = ilk garaj alfabetik) ===
Negatif = referans garaj daha kotu, pozitif = referans daha iyi

✓ C(GARAJ)[T.Edirnekapı]                              k=+0.1435  p=0.0060
✓ C(GARAJ)[T.Hasanpaşa]                               k=+0.1658  p=0.0070
✓ C(GARAJ)[T.IKITELLIGARAJI]                          k=-0.1550  p=0.0052
✓ C(GARAJ)[T.IKITELLIISLETTIRMEGARAJI2]               k=-0.2205  p=0.0001
✓ C(GARAJ)[T.KURTKÖY]                                 k=+0.1922  p=0.0003
✓ C(GARAJ)[T.Kağıthane]                               k=+0.2457  p=0.0001
✓ C(GARAJ)[T.SULTANGAZIGARAJI]                        k=-0.1929  p=0.0002
  C(GARAJ)[T.Sarıgazi]                                k=-0.0275  p=0.6594
✓ C(GARAJ)[T.Topkapı]                        

---

## 7. Yaş Stratify
Aynı yaş grubu içinde garaj farkı kalıyor mu? Yaş confounder'ından bağımsız etki testi.

In [7]:
# BOLUM 7: Yaş Gruplari Icinde Garaj Etkisi (Stratify)
# Soru: Yas confounder cikartilirsa garaj farki kalir mi?
# Yontem: Aynı yas grubu icinde garaj × ciddiyet karsilastir

arac_full['yas_grup'] = pd.cut(arac_full['yas'],
    bins=[0, 5, 10, 15, 25],
    labels=['Yeni (0-5)', 'Genc (6-10)', 'Orta (11-15)', 'Yasli (16+)'])

print('=== YAS GRUBU × GARAJ ORT_SKOR ===')
print('Eger yas grubu icinde garaj farki kaliyorsa, garaj bagimsiz etki var')
print()

for grup in ['Yeni (0-5)', 'Genc (6-10)', 'Orta (11-15)', 'Yasli (16+)']:
    sub = arac_full[arac_full['yas_grup'] == grup]
    if len(sub) < 50: continue

    g_skor = sub.groupby('GARAJ').agg(
        n=('KAPINO','count'),
        skor=('ort_skor','mean'),
    ).query('n >= 10').sort_values('skor', ascending=False)

    if len(g_skor) < 3: continue
    print(f'--- {grup} (n_arac={len(sub):,}) ---')
    for garaj, row in g_skor.iterrows():
        print(f'  {garaj[:30]:30s}: n={row.n:4.0f}, ort_skor={row.skor:.3f}')

    # ANOVA bu yas grubu icinde
    gruplar = [g['ort_skor'].dropna().values for _, g in sub.groupby('GARAJ') if len(g) >= 10]
    if len(gruplar) >= 3:
        f, p = stats.f_oneway(*gruplar)
        print(f'  ANOVA F={f:.2f}, p={p:.4f}  {"← ANLAMLI" if p<0.05 else ""}')
    print()


=== YAS GRUBU × GARAJ ORT_SKOR ===
Eger yas grubu icinde garaj farki kaliyorsa, garaj bagimsiz etki var

--- Yeni (0-5) (n_arac=406) ---
  Hasanpaşa                     : n= 119, ort_skor=3.989
  Edirnekapı                    : n= 132, ort_skor=3.785
  Topkapı                       : n= 151, ort_skor=2.068
  ANOVA F=432.48, p=0.0000  ← ANLAMLI

--- Genc (6-10) (n_arac=551) ---
  IKITELLIISLETTIRMEGARAJI2     : n=  45, ort_skor=3.628
  Hasanpaşa                     : n= 125, ort_skor=3.583
  IKITELLIGARAJI                : n= 381, ort_skor=3.466
  ANOVA F=1.66, p=0.1906  

--- Orta (11-15) (n_arac=1,862) ---
  KURTKÖY                       : n= 346, ort_skor=3.853
  Kağıthane                     : n= 244, ort_skor=3.835
  Hasanpaşa                     : n=  75, ort_skor=3.706
  Sarıgazi                      : n= 180, ort_skor=3.625
  Anadolu                       : n=  90, ort_skor=3.611
  IKITELLIISLETTIRMEGARAJI2     : n= 285, ort_skor=3.463
  SULTANGAZIGARAJI              : n= 413, o

---

## 8. Spesifik Anomali Tespiti
Yüksek lift garaj-marka kombinasyonları + cascade oranıyla karşılaştırma.

In [8]:
# BOLUM 8: Spesifik Anomali — Yuksek Lift Garaj-Marka Kombinasyonu
# Soru: Hangi spesifik garaj-marka cifti anormal yuksek arıza?

# Bolum 2'deki matrix kullanalim
# Yuksek lift = anormal kotu
anomali = matrix[matrix['ciddi_lift'] > 1.20].sort_values('ciddi_lift', ascending=False)
print(f'Anomali kombinasyon sayisi (lift>1.20): {len(anomali)}')
print()
print('=== EN COK SAPMA GOSTEREN GARAJ-MARKA CIFTLERI ===')
print(anomali.head(15)[['GARAJ','MARKA','n','ciddi','marka_genel','ciddi_lift']].round(3).to_string(index=False))

# Bu kombinasyonlarda cascade var mi (tamir kalitesi kotu mu)?
print()
print('=== ANOMALI CIFTLERDE CASCADE ORANI ===')
print('Eger cascade da yuksekse, tamir kalitesi suphesi guclenir')
print()
anomali_top10 = anomali.head(10)
for _, row in anomali_top10.iterrows():
    sub = df[(df['GARAJ'] == row['GARAJ']) & (df['MARKA'] == row['MARKA'])]
    if len(sub) < 30: continue
    cas_oran = sub['cascade_24s'].mean()
    print(f'  {row["GARAJ"][:25]:25s} × {row["MARKA"]:12s}: ciddi_lift={row["ciddi_lift"]:.2f}, cascade_24s={cas_oran:.3f} (filo:{toplam_cas24:.3f})')


Anomali kombinasyon sayisi (lift>1.20): 3

=== EN COK SAPMA GOSTEREN GARAJ-MARKA CIFTLERI ===
                    GARAJ  MARKA    n  ciddi  marka_genel  ciddi_lift
IKITELLIISLETTIRMEGARAJI2    BMC 1116  0.469        0.361       1.298
               Edirnekapı   AKIA 1771  0.420        0.329       1.279
                Hasanpaşa OTOKAR 2924  0.481        0.387       1.242

=== ANOMALI CIFTLERDE CASCADE ORANI ===
Eger cascade da yuksekse, tamir kalitesi suphesi guclenir

  IKITELLIISLETTIRMEGARAJI2 × BMC         : ciddi_lift=1.30, cascade_24s=0.152 (filo:0.149)
  Edirnekapı                × AKIA        : ciddi_lift=1.28, cascade_24s=0.127 (filo:0.149)
  Hasanpaşa                 × OTOKAR      : ciddi_lift=1.24, cascade_24s=0.201 (filo:0.149)


---

## 9. ML Feature Türetme
10 garaj bazlı feature kandidatı.

In [9]:
# BOLUM 9: ML Feature Türetme
# Garaj bazli feature kandidatlari

# Her arıza icin: o aracın garajinin pattern'ı
garaj_skor_map = garaj_profil.set_index('GARAJ').to_dict()

df_feat = df.copy()
df_feat['garaj_ort_skor'] = df_feat['GARAJ'].map(garaj_skor_map.get('ort_skor', {}))
df_feat['garaj_ciddi_oran'] = df_feat['GARAJ'].map(garaj_skor_map.get('ciddi_oran', {}))
df_feat['garaj_ariza_per_1000_sefer'] = df_feat['GARAJ'].map(garaj_skor_map.get('ariza_per_1000_sefer', {}))

# Garaj × Marka lift
matrix_lookup = matrix.set_index(['GARAJ','MARKA'])['ciddi_lift'].to_dict()
df_feat['garaj_marka_lift'] = df_feat.apply(
    lambda r: matrix_lookup.get((r['GARAJ'], r['MARKA']), 1.0), axis=1
)

# Garaj cascade orani
garaj_cas_map = garaj_cas.set_index('GARAJ')['cascade_24s_oran'].to_dict()
df_feat['garaj_cascade_oran'] = df_feat['GARAJ'].map(garaj_cas_map).fillna(toplam_cas24)

# Garaj cascade lift
garaj_cas_lift_map = garaj_cas.set_index('GARAJ')['cas24s_lift'].to_dict()
df_feat['garaj_cas_lift'] = df_feat['GARAJ'].map(garaj_cas_lift_map).fillna(1.0)

# Garaj filo yas ortalamasi (Analiz 3'ten)
garaj_yas_map = garaj_profil.set_index('GARAJ')['ort_yas'].to_dict()
df_feat['garaj_filo_yas'] = df_feat['GARAJ'].map(garaj_yas_map)

# Garaj × Sistem lift
sistem_matrix_lookup = matrix_s.set_index(['GARAJ','ARIZAUSTKODTANIM'])['lift'].to_dict()
df_feat['garaj_sistem_lift'] = df_feat.apply(
    lambda r: sistem_matrix_lookup.get((r['GARAJ'], r['ARIZAUSTKODTANIM']), 1.0), axis=1
)

# Garaj uzman skoru (sistem sayisi)
uzman_map = uzmanlik.set_index('GARAJ')['uzman_sistem_n'].to_dict()
sorun_map = sorun.set_index('GARAJ')['sorunlu_sistem_n'].to_dict()
df_feat['garaj_uzman_skor'] = df_feat['GARAJ'].map(uzman_map).fillna(0)
df_feat['garaj_sorun_skor'] = df_feat['GARAJ'].map(sorun_map).fillna(0)

print('Yeni feature\'lar:')
features = [
    'garaj_ort_skor', 'garaj_ciddi_oran', 'garaj_ariza_per_1000_sefer',
    'garaj_marka_lift', 'garaj_cascade_oran', 'garaj_cas_lift',
    'garaj_filo_yas', 'garaj_sistem_lift', 'garaj_uzman_skor', 'garaj_sorun_skor'
]
for f in features:
    print(f'  {f}')
print(f'\nFeature seti: {len(df_feat):,} satir')


Yeni feature'lar:
  garaj_ort_skor
  garaj_ciddi_oran
  garaj_ariza_per_1000_sefer
  garaj_marka_lift
  garaj_cascade_oran
  garaj_cas_lift
  garaj_filo_yas
  garaj_sistem_lift
  garaj_uzman_skor
  garaj_sorun_skor

Feature seti: 58,559 satir


---

## 10. Korelasyon + ANOVA + Bant Analizi
Her feature için Pearson + Spearman + ANOVA. **Karar veriye göre.**

In [10]:
# BOLUM 10: Korelasyon + ANOVA + Bant Analizi

features = [
    'garaj_ort_skor', 'garaj_ciddi_oran', 'garaj_ariza_per_1000_sefer',
    'garaj_marka_lift', 'garaj_cascade_oran', 'garaj_cas_lift',
    'garaj_filo_yas', 'garaj_sistem_lift', 'garaj_uzman_skor', 'garaj_sorun_skor'
]

print('=== FEATURE × CIDDIYET KORELASYON ===')
print(f'{"Feature":30s}  Pearson_r  p        Spearman_rs  CiddiAriza_r')

sonuclar = []
for f in features:
    if f not in df_feat.columns: continue
    valid = df_feat.dropna(subset=[f])
    r, p = stats.pearsonr(valid[f], valid['ciddiyet_skoru'])
    rs, ps = stats.spearmanr(valid[f], valid['ciddiyet_skoru'])
    rc, pc = stats.pointbiserialr(valid['ciddi_ariza'], valid[f])
    sonuclar.append({'feature': f, 'r': r, 'p': p, 'rs': rs, 'rc': rc, 'abs_r': abs(r)})
    print(f'{f:30s}  {r:+.4f}    {p:.4f}   {rs:+.4f}      {rc:+.4f}')

korr_df = pd.DataFrame(sonuclar).sort_values('abs_r', ascending=False)

print()
print('=== VERI BAZLI KARAR ===')
print('GUCLU FEATURE\'LAR (|r| > 0.05 ve p < 0.05):')
guclu = korr_df[(korr_df['abs_r'] > 0.05) & (korr_df['p'] < 0.05)]
if len(guclu) > 0:
    for _, r in guclu.iterrows():
        print(f'  + {r.feature}  (r={r.r:+.4f})')
else:
    print('  Hicbiri |r|>0.05 esigini gecmiyor.')

print()
print('ZAYIF FEATURE\'LAR:')
zayif = korr_df[korr_df['abs_r'] <= 0.05]
for _, r in zayif.iterrows():
    print(f'  - {r.feature}  (r={r.r:+.4f})')

# Bant analizi en guclu feature icin
en_guclu = korr_df.iloc[0]['feature']
print(f'\n=== BANT ANALIZI ({en_guclu}) ===')
df_feat[f'{en_guclu}_bant'] = pd.qcut(df_feat[en_guclu], 4, labels=['Q1 (en dusuk)','Q2','Q3','Q4 (en yuksek)'], duplicates='drop')
print(df_feat.groupby(f'{en_guclu}_bant', observed=False).agg(
    n=(en_guclu,'count'),
    ort_skor=('ciddiyet_skoru','mean'),
    ciddi=('ciddi_ariza','mean'),
).round(3).to_string())

# ANOVA
gruplar = [g['ciddiyet_skoru'].values for _, g in df_feat.groupby(f'{en_guclu}_bant', observed=False) if len(g) >= 50]
if len(gruplar) >= 2:
    f, p = stats.f_oneway(*gruplar)
    print(f'\nANOVA F={f:.2f}, p={p:.6f}')
    if p < 0.05:
        print('VERIYE GORE: Bantlar arasi ANLAMLI fark — tree model icin uygun.')


=== FEATURE × CIDDIYET KORELASYON ===
Feature                         Pearson_r  p        Spearman_rs  CiddiAriza_r
garaj_ort_skor                  +0.1318    0.0000   +0.1294      +0.0676
garaj_ciddi_oran                +0.1001    0.0000   +0.0999      +0.0893
garaj_ariza_per_1000_sefer      +0.0246    0.0000   +0.0625      +0.0518
garaj_marka_lift                +0.0844    0.0000   +0.0912      +0.1038
garaj_cascade_oran              +0.0316    0.0000   +0.0676      +0.0590
garaj_cas_lift                  +0.0316    0.0000   +0.0676      +0.0590
garaj_filo_yas                  +0.0436    0.0000   +0.0155      +0.0019
garaj_sistem_lift               +0.2131    0.0000   +0.2155      +0.1787
garaj_uzman_skor                -0.0189    0.0000   -0.0387      -0.0328
garaj_sorun_skor                +0.0439    0.0000   +0.0775      +0.0557

=== VERI BAZLI KARAR ===
GUCLU FEATURE'LAR (|r| > 0.05 ve p < 0.05):
  + garaj_sistem_lift  (r=+0.2131)
  + garaj_ort_skor  (r=+0.1318)
  + garaj_ciddi_o

---

## 11. Time-Based Leakage Testi (KRİTİK — Analiz 2 Dersi)
Global garaj feature'ları leakage'dan mı? Train→Test karşılaştırma + stabilite.

In [11]:
# BOLUM 11: Time-Based Leakage Testi (Analiz 2 dersi)
# Soru: garaj_ort_skor gibi global feature'lar leakage'dan mi geliyor?
# Yontem: Train ilk yari, test sonraki yari

df_sort = df.sort_values('OLAYTARIHI').reset_index(drop=True)
median_t = df_sort['OLAYTARIHI'].median()
train = df_sort[df_sort['OLAYTARIHI'] < median_t].copy()
test  = df_sort[df_sort['OLAYTARIHI'] >= median_t].copy()

print(f'Train: {len(train):,} ariza  Test: {len(test):,} ariza')

# Garaj kapsami
train_garajlar = set(train['GARAJ'])
test_garajlar = set(test['GARAJ'])
ortak = train_garajlar & test_garajlar
yeni = test_garajlar - train_garajlar
print(f'Test\'te ortak garaj: {len(ortak)}, yeni garaj: {len(yeni)}')

# Train'den garaj profilini cikar
train_garaj_skor = train.groupby('GARAJ').agg(
    train_skor=('ciddiyet_skoru', 'mean'),
    train_ciddi=('ciddi_ariza', 'mean'),
    train_cascade=('cascade_24s', 'mean'),
    train_n=('ciddi_ariza', 'count'),
).reset_index()

# Test'e merge
test_w = test.merge(train_garaj_skor, on='GARAJ', how='left')
test_w_clean = test_w.dropna(subset=['train_skor'])
print(f'Test\'te train_skor atanan: {len(test_w_clean):,}')

# Time-aware korelasyonlar
r_skor, p_skor = stats.pearsonr(test_w_clean['train_skor'], test_w_clean['ciddiyet_skoru'])
r_cas, p_cas = stats.pearsonr(test_w_clean['train_cascade'], test_w_clean['cascade_24s'])

# Full-data karsilastirma
df_full_w = df.merge(df.groupby('GARAJ')['ciddiyet_skoru'].mean().reset_index().rename(columns={'ciddiyet_skoru':'g_skor'}), on='GARAJ')
r_full_skor, _ = stats.pearsonr(df_full_w['g_skor'], df_full_w['ciddiyet_skoru'])

df_full_c = df.merge(df.groupby('GARAJ')['cascade_24s'].mean().reset_index().rename(columns={'cascade_24s':'g_cas'}), on='GARAJ')
r_full_cas, _ = stats.pearsonr(df_full_c['g_cas'], df_full_c['cascade_24s'])

print()
print('=== LEAKAGE TESTI: FULL-DATA vs TIME-AWARE ===')
print(f'{"Feature":25s}  Full-data r  Time-aware r  Düşüş %')
dususp1 = (1 - r_skor/r_full_skor)*100 if r_full_skor>0 else 0
dususp2 = (1 - r_cas/r_full_cas)*100 if r_full_cas>0 else 0
print(f'{"garaj_ort_skor":25s}  {r_full_skor:+.4f}      {r_skor:+.4f}      %{dususp1:.0f}')
print(f'{"garaj_cascade_oran":25s}  {r_full_cas:+.4f}      {r_cas:+.4f}      %{dususp2:.0f}')

# Garaj profil stabilitesi
test_garaj_skor = test.groupby('GARAJ').agg(
    test_skor=('ciddiyet_skoru','mean'),
    test_n=('ciddi_ariza','count'),
).reset_index()
ortak_df = train_garaj_skor.merge(test_garaj_skor, on='GARAJ')
ortak_df = ortak_df[(ortak_df['train_n'] >= 30) & (ortak_df['test_n'] >= 30)]
if len(ortak_df) >= 5:
    r_stab, p_stab = stats.pearsonr(ortak_df['train_skor'], ortak_df['test_skor'])
    print()
    print(f'=== GARAJ PROFIL STABILITESI ===')
    print(f'n_garaj: {len(ortak_df)}')
    print(f'Pearson r = {r_stab:+.4f}, p = {p_stab:.4f}')
    if r_stab > 0.5:
        print('STABIL: Garaj profili zamanda korunuyor — gercek pattern.')
    elif r_stab > 0.2:
        print('ORTA STABIL: Pattern var ama gurulu.')
    else:
        print('STABIL DEGIL: Garaj skoru degisken — leakage suphesi yuksek.')

# Karar
print()
print('=== VERI BAZLI KARAR ===')
if abs(r_skor) > 0.15:
    print(f'Time-aware r={r_skor:+.3f} GUCLU → garaj feature\'lari ML icin guvenilir.')
elif abs(r_skor) > 0.05:
    print(f'Time-aware r={r_skor:+.3f} ZAYIF ama anlamli → orta degerde feature.')
else:
    print(f'Time-aware r={r_skor:+.3f} cok zayıf → leakage dominant, ML guvensiz.')


Train: 29,279 ariza  Test: 29,280 ariza
Test'te ortak garaj: 12, yeni garaj: 0
Test'te train_skor atanan: 29,280

=== LEAKAGE TESTI: FULL-DATA vs TIME-AWARE ===
Feature                    Full-data r  Time-aware r  Düşüş %
garaj_ort_skor             +0.1318      +0.1272      %3
garaj_cascade_oran         +0.0997      +0.0976      %2

=== GARAJ PROFIL STABILITESI ===
n_garaj: 12
Pearson r = +0.9403, p = 0.0000
STABIL: Garaj profili zamanda korunuyor — gercek pattern.

=== VERI BAZLI KARAR ===
Time-aware r=+0.127 ZAYIF ama anlamli → orta degerde feature.


---

## 12. Vaka Analizi — En İyi/Kötü Garaj-Marka Kombinasyonları
Kompozit skor (lift + cascade) ile en iyi/kötü çiftler. Topkapı/Anadolu özel detay.

In [12]:
# BOLUM 12: Vaka Analizi — En Iyi/Kotu Garaj-Marka Kombinasyonlari

# En iyi garaj-marka cifti: dusuk ciddi, dusuk cascade
matrix_full = matrix.copy()
matrix_full['cascade_oran'] = matrix_full.apply(
    lambda r: df[(df['GARAJ']==r['GARAJ']) & (df['MARKA']==r['MARKA'])]['cascade_24s'].mean(), axis=1
)

# En iyi: dusuk ciddi_lift + dusuk cascade
print('=== EN IYI GARAJ-MARKA CIFTLERI (dusuk lift + dusuk cascade) ===')
matrix_full['kompozit_skor'] = matrix_full['ciddi_lift'] + matrix_full['cascade_oran']
print(matrix_full.nsmallest(10, 'kompozit_skor')[
    ['GARAJ','MARKA','n','ciddi','marka_genel','ciddi_lift','cascade_oran']
].round(3).to_string(index=False))

print()
print('=== EN KOTU GARAJ-MARKA CIFTLERI (yuksek lift + yuksek cascade) ===')
print(matrix_full.nlargest(10, 'kompozit_skor')[
    ['GARAJ','MARKA','n','ciddi','marka_genel','ciddi_lift','cascade_oran']
].round(3).to_string(index=False))

# Onceden bilinen vakalar — Topkapi (en genc), Anadolu (en yasli) icin profil
print()
print('=== ONCEDEN BILINEN GARAJ DETAYI ===')
for garaj in ['Topkapı', 'Anadolu', 'Hasanpaşa', 'IKITELLIISLETTIRMEGARAJI2']:
    sub = arac_full[arac_full['GARAJ'] == garaj]
    if len(sub) < 10: continue
    print(f'\n--- {garaj} ---')
    print(f'  Arac sayisi: {len(sub)}')
    print(f'  Ort. yas: {sub["yas"].mean():.1f}')
    print(f'  Ort. egim_puan: {sub["ort_egim"].mean():.1f}')
    print(f'  Ort. ciddiyet: {sub["ort_skor"].mean():.3f}')
    print(f'  Cascade orani: {sub["cascade_oran"].mean():.3f}')
    print(f'  Ariza/1000 sefer: {sub["ariza_n"].sum() / (sub["toplam_sefer"].sum()/1000):.2f}')


=== EN IYI GARAJ-MARKA CIFTLERI (dusuk lift + dusuk cascade) ===
                    GARAJ    MARKA    n  ciddi  marka_genel  ciddi_lift  cascade_oran
                  Topkapı     AKIA  801  0.126        0.329       0.384         0.081
                Kağıthane   KARSAN 2294  0.334        0.411       0.812         0.088
                    Yunus   OTOKAR 2880  0.314        0.387       0.812         0.108
                  Anadolu MERCEDES 4900  0.333        0.387       0.860         0.120
           IKITELLIGARAJI      BMC 3559  0.327        0.361       0.907         0.084
IKITELLIISLETTIRMEGARAJI2 MERCEDES  176  0.347        0.387       0.895         0.119
                 Sarıgazi   OTOKAR 1910  0.358        0.387       0.924         0.098
         SULTANGAZIGARAJI   OTOKAR 5445  0.377        0.387       0.975         0.141
                  KURTKÖY   KARSAN 2914  0.419        0.411       1.020         0.147
                  KURTKÖY   OTOKAR 4351  0.397        0.387       1.027    

---

## 13. Operasyonel Karar Mekanizması
Hangi araç hangi garaja yönlendirilmeli? Marka × sistem × cascade tabanlı öneri.

In [13]:
# BOLUM 13: Operasyonel Oneri
# Karar mekanizmasi: hangi arac hangi garaja gitsin?

# 1. Her marka için "en iyi" garaj (lift en dusuk + n yeterli)
print('=== HER MARKA ICIN EN UYUMLU GARAJ ===')
for marka in top_markalar:
    sub = matrix[matrix['MARKA'] == marka].copy()
    if len(sub) < 2: continue
    en_iyi = sub.nsmallest(1, 'ciddi_lift')
    en_kotu = sub.nlargest(1, 'ciddi_lift')

    if len(en_iyi) > 0 and len(en_kotu) > 0:
        ei = en_iyi.iloc[0]
        ek = en_kotu.iloc[0]
        print(f'{marka:12s}: EN IYI {ei["GARAJ"][:25]:25s} (lift {ei["ciddi_lift"]:.2f}) | EN KOTU {ek["GARAJ"][:25]:25s} (lift {ek["ciddi_lift"]:.2f})')

# 2. Sistem bazli garaj uzmanlik
print()
print('=== SISTEM BAZLI EN UZMAN GARAJ ===')
for sis in df['ARIZAUSTKODTANIM'].value_counts().head(8).index:
    sub_s = matrix_s[matrix_s['ARIZAUSTKODTANIM'] == sis].copy()
    if len(sub_s) < 3: continue
    en_iyi = sub_s.nsmallest(1, 'lift')
    if len(en_iyi) > 0:
        ei = en_iyi.iloc[0]
        print(f'{str(sis)[:35]:35s}: {ei["GARAJ"][:25]:25s} (lift {ei["lift"]:.2f})')

# 3. Karar tablosu
print()
print('=== OPERASYONEL KARAR TABLOSU ===')
karar = []
for garaj in garaj_profil_clean['GARAJ']:
    sub = garaj_profil_clean[garaj_profil_clean['GARAJ'] == garaj].iloc[0]
    cas_lift = garaj_cas.set_index('GARAJ').get('cas24s_lift', {}).get(garaj, 1.0) if garaj in garaj_cas['GARAJ'].values else 1.0
    karar.append({
        'GARAJ': garaj,
        'n_arac': sub['n_arac'],
        'ort_yas': sub['ort_yas'],
        'ort_skor': sub['ort_skor'],
        'cascade_lift': cas_lift,
        'durum': 'ALARM' if cas_lift > 1.15 else ('IYI' if cas_lift < 0.85 else 'NORMAL')
    })
karar_df = pd.DataFrame(karar).sort_values('cascade_lift', ascending=False)
print(karar_df.round(2).to_string(index=False))

print()
print('=== AKSIYON ONERILERI ===')
print('1. Cascade lift > 1.15 garajlar tamir kalitesi denetlenmeli (veri sebep gostermiyor, ama uyari).')
print('2. Marka bazli rotasyon: her marka kendi en uyumlu garajinda toplanmali (matrix uyum).')
print('3. Sistem uzmanligi: belirli sistemler icin "uzman garaj" yonlendirme.')
print('4. NOT: Bakim turu (degisim/tamir) verisi yok, sebep yorumu sinirli.')


=== HER MARKA ICIN EN UYUMLU GARAJ ===
MERCEDES    : EN IYI Anadolu                   (lift 0.86) | EN KOTU IKITELLIGARAJI            (lift 1.14)
OTOKAR      : EN IYI Yunus                     (lift 0.81) | EN KOTU Hasanpaşa                 (lift 1.24)
KARSAN      : EN IYI Kağıthane                 (lift 0.81) | EN KOTU IKITELLIISLETTIRMEGARAJI2 (lift 1.08)
BMC         : EN IYI IKITELLIGARAJI            (lift 0.91) | EN KOTU IKITELLIISLETTIRMEGARAJI2 (lift 1.30)
AKIA        : EN IYI Topkapı                   (lift 0.38) | EN KOTU Edirnekapı                (lift 1.28)

=== SISTEM BAZLI EN UZMAN GARAJ ===
SOĞUTMA SİSTEMİ ARIZASI            : Sarıgazi                  (lift 0.49)
KAPI ARIZALARI                     : Kağıthane                 (lift 0.31)
ELEKTRİK SİSTEMİ ARIZALARI         : Topkapı                   (lift 0.23)
MOTOR ARIZALARI                    : IKITELLIGARAJI            (lift 0.63)
KAROSER ARIZALARI                  : Edirnekapı                (lift 0.77)
KLİMA SİSTEMİ 

---

## 14. Topkapi Tahsis Yanliligi (Selection Bias)

Soru: Topkapi'nin "en iyi" gorunmesi tamir kalitesi mi, yoksa "yeni araclar oraya gonderiliyor" secim yanliligi mi?

Yontem:
1. Her garaj icin yas dagilimi (mean, median, P25, P75) ve yeni arac (<=3 yas) orani
2. Yeni arac orani garajlar arasi anlamli farkli mi? (Chi-square)
3. Eger Topkapi'da yeni arac orani orantisizsa -> tahsis yanliligi kaniti
4. Karsi-kontrol: ayni yas bandi icinde Topkapi hala daha iyi mi? (Bolum 7 zaten kanitladi)


In [14]:
# BOLUM 14: Topkapi Tahsis Yanliligi Testi
# Soru: Yeni araclar orantisiz bicimde Topkapi'ya mi tahsis ediliyor?

from scipy.stats import chi2_contingency

# Garaj basina yas profili
yas_profil = arac_full.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_mean=('yas','mean'),
    yas_median=('yas','median'),
    yas_p25=('yas', lambda s: s.quantile(0.25)),
    yas_p75=('yas', lambda s: s.quantile(0.75)),
).round(2).sort_values('yas_mean')

arac_full['yas_band'] = pd.cut(arac_full['yas'], bins=[-1, 3, 10, 100], labels=['Yeni','Orta','Yasli'])
band_dist = arac_full.groupby(['GARAJ','yas_band']).size().unstack(fill_value=0)
band_dist['toplam'] = band_dist.sum(axis=1)
band_dist['yeni_oran'] = (band_dist['Yeni'] / band_dist['toplam']).round(3)
band_dist['yasli_oran'] = (band_dist['Yasli'] / band_dist['toplam']).round(3)

print('=== GARAJ x YAS PROFILI ===')
print(yas_profil.to_string())
print()
print('=== YENI/YASLI ARAC ORANI ===')
print(band_dist[['Yeni','Orta','Yasli','toplam','yeni_oran','yasli_oran']].sort_values('yeni_oran', ascending=False).to_string())

contingency = band_dist[['Yeni','Orta','Yasli']].values
chi2, p_chi, dof, exp = chi2_contingency(contingency)
print()
print('=== CHI-SQUARE: Yas band x Garaj ===')
print(f'chi2 = {chi2:.2f}, p = {p_chi:.6f}, dof = {dof}')
if p_chi < 0.001:
    print('SONUC: Garajlar arasi yas dagilimi ANLAMLI farkli -> tahsis politikasi heterojen')
else:
    print('SONUC: Garajlar arasi yas dagilimi homojen')

topkapi_key = [g for g in band_dist.index if 'TOPKAPI' in g.upper()]
if topkapi_key:
    tk_garaj = topkapi_key[0]
    topkapi_yeni = band_dist.loc[tk_garaj, 'yeni_oran']
    ort_yeni = band_dist['yeni_oran'].drop(tk_garaj).mean()
    print()
    print('=== TOPKAPI TAHSIS YANLILIGI ===')
    print(f'Topkapi ({tk_garaj}) yeni arac orani:     %{topkapi_yeni*100:.1f}')
    print(f'Diger garajlar ortalama:                     %{ort_yeni*100:.1f}')
    if ort_yeni > 0:
        print(f'Fark (kat):                                 {topkapi_yeni/ort_yeni:.2f}x')
    if topkapi_yeni > ort_yeni * 2:
        print('  -> Topkapi yeni araclari ORANTISIZ aliyor (tahsis yanliligi pozitif)')
        print('  -> Bolum 7 stratify analizi yine de garaj farkini koruyor -> "saf garaj etkisi" var')
    else:
        print('  -> Topkapi yeni arac orani diger garajlardan cok farkli degil')

print()
print('=== YORUM ===')
print('Tahsis yanliligi varsa: regresyon yasi kontrol ediyor ama personel/yedek parca confounder kalir.')
print('Bolum 7 stratify (F=432 Yeni grupta) bagimsiz garaj etkisini hala destekliyor.')


=== GARAJ x YAS PROFILI ===
                           n_arac  yas_mean  yas_median  yas_p25  yas_p75
GARAJ                                                                    
Topkapı                       152      1.12         1.0      1.0      1.0
Hasanpaşa                     319      8.03        10.0      3.0     10.0
IKITELLIGARAJI                429      9.23         8.0      8.0      8.0
IKITELLIISLETTIRMEGARAJI2     330     11.32        12.0     11.0     12.0
Sarıgazi                      184     11.78        12.0     12.0     12.0
Edirnekapı                    381     11.83        16.0      3.0     17.0
Kağıthane                     244     12.00        12.0     12.0     12.0
Yunus                         229     12.00        12.0     12.0     12.0
KURTKÖY                       346     12.07        12.0     12.0     12.0
SULTANGAZIGARAJI              413     12.24        12.0     12.0     12.0
Anadolu                       353     17.36        19.0     13.0     19.0
Şahinkaya 

---

## 15. Bootstrap CI - Kucuk Garaj Orneklem Guveni

Soru: Kucuk arac sayili garajlarin garaj_ort_skor degerleri guvenilir mi yoksa istatistiksel gurultu mu?

Yontem:
1. Her garaj icin bootstrap 1000 orneklem
2. ort_skor icin %95 guven araligi
3. Kucuk n'li garajlarda CI ne kadar genis?
4. Topkapi-Hasanpasa CI ortusuyor mu? (fark garantili mi?)


In [15]:
# BOLUM 15: Bootstrap CI
import numpy as np
np.random.seed(42)
N_BOOT = 1000

bootstrap_results = []
for garaj, grp in arac_full.groupby('GARAJ'):
    n = len(grp)
    skor_mean = grp['ort_skor'].mean()
    vals = grp['ort_skor'].values
    boot_means = np.array([
        np.random.choice(vals, size=n, replace=True).mean()
        for _ in range(N_BOOT)
    ])
    ci_low = np.percentile(boot_means, 2.5)
    ci_high = np.percentile(boot_means, 97.5)
    bootstrap_results.append({
        'GARAJ': garaj, 'n': n, 'ort_skor': round(skor_mean, 3),
        'ci_low': round(ci_low, 3), 'ci_high': round(ci_high, 3),
        'ci_width': round(ci_high - ci_low, 3)
    })

boot_df = pd.DataFrame(bootstrap_results).sort_values('ort_skor')

print('=== GARAJ x ORT_SKOR %95 BOOTSTRAP CI ===')
print(boot_df.to_string(index=False))

print()
print('=== GUVEN SIRALAMASI ===')
guvensiz = boot_df[boot_df['ci_width'] > 0.3]
guvenli = boot_df[boot_df['ci_width'] <= 0.3]
print(f'Guvenli (CI width <= 0.3): {len(guvenli)} garaj')
if len(guvenli) > 0:
    print(f'  En guvenli: {boot_df.loc[boot_df["ci_width"].idxmin(), "GARAJ"]} (width={boot_df["ci_width"].min():.3f})')
print(f'Gri zon (CI width > 0.3):  {len(guvensiz)} garaj')
if len(guvensiz) > 0:
    print(f'  Bu garajlar icin: {guvensiz["GARAJ"].tolist()}')

tk_keys = [g for g in boot_df['GARAJ'].values if 'TOPKAPI' in g.upper()]
hp_keys = [g for g in boot_df['GARAJ'].values if 'HASANPASA' in g.upper()]
if tk_keys and hp_keys:
    tk = boot_df[boot_df['GARAJ']==tk_keys[0]].iloc[0]
    hp = boot_df[boot_df['GARAJ']==hp_keys[0]].iloc[0]
    overlap = (tk['ci_high'] >= hp['ci_low']) and (hp['ci_high'] >= tk['ci_low'])
    print()
    print('=== TOPKAPI vs HASANPASA CI ORTUSUYOR MU? ===')
    print(f'Topkapi:    [{tk["ci_low"]:.3f}, {tk["ci_high"]:.3f}]')
    print(f'Hasanpasa:  [{hp["ci_low"]:.3f}, {hp["ci_high"]:.3f}]')
    if overlap:
        print('  -> CI ORTUSUYOR! Iki garaj arasi fark istatistiksel olarak garanti degil')
    else:
        print('  -> CI ORTUSMUYOR -> Topkapi-Hasanpasa farki GARANTILI')

print()
print('=== YORUM ===')
print('Dar CI olan garaj feature degerlerine guven (ML kullaniminda).')
print('Gri zon garajlari icin: feature belirsiz olarak isaretle veya ek veri bekle.')


=== GARAJ x ORT_SKOR %95 BOOTSTRAP CI ===
                    GARAJ   n  ort_skor  ci_low  ci_high  ci_width
                  Topkapı 152     2.082   1.965    2.200     0.235
                    Yunus 229     3.359   3.272    3.447     0.175
         SULTANGAZIGARAJI 413     3.437   3.382    3.495     0.113
           IKITELLIGARAJI 429     3.466   3.379    3.541     0.162
IKITELLIISLETTIRMEGARAJI2 330     3.486   3.428    3.539     0.111
                  Anadolu 353     3.566   3.503    3.627     0.124
                 Sarıgazi 184     3.622   3.505    3.740     0.235
               Edirnekapı 381     3.751   3.696    3.806     0.110
                Hasanpaşa 319     3.763   3.717    3.811     0.094
                Kağıthane 244     3.835   3.730    3.933     0.203
                  KURTKÖY 346     3.853   3.800    3.908     0.109
                Şahinkaya 129     4.065   3.916    4.214     0.298

=== GUVEN SIRALAMASI ===
Guvenli (CI width <= 0.3): 12 garaj
  En guvenli: Hasanpaşa (

---

## 16. ARACTIPI Confounder - Metrobus vs Otobus Etkisi

Kritik bulgu: ariza_model.csv ARACTIPI icermiyor. ariza_temiz.csv'den merge:
- Edirnekapi 381 arac TAMAMI METROBUS
- Hasanpasa 319 arac TAMAMI METROBUS
- Topkapi 151 arac TAMAMI OTOBUS
- Anadolu, SULTANGAZI = karisik

Soru: Garaj farkinin ne kadari ARACTIPI'nden geliyor? M4'e ARACTIPI ekle (M5).
Beklenen: Hasanpasa & Edirnekapi katsayilari M5'te DUSER (metrobus etkisi yutulur).


In [16]:
# BOLUM 16: ARACTIPI Confounder Kontrolu (M5)
import statsmodels.api as sm
from statsmodels.formula.api import ols

OTOBUS = "Otob\u00fcs"
METROBUS = "Metrob\u00fcs"

at = pd.read_csv(r'panel_data\\temiz_veri\\ariza_temiz.csv', usecols=['KAPINO','ARACTIPI'], low_memory=False)
arac_aractip = at.groupby('KAPINO')['ARACTIPI'].agg(lambda s: s.mode()[0] if len(s.mode()) else 'X').reset_index()
print('ARACTIPI dagilimi (arac bazinda):', arac_aractip['ARACTIPI'].value_counts().to_dict())

arac_full2 = arac_full.merge(arac_aractip, on='KAPINO', how='left')
arac_full2 = arac_full2[arac_full2['ARACTIPI'].isin([OTOBUS, METROBUS])].copy()
print(f'M5 regresyon seti: {len(arac_full2):,} arac')
print(f'  Otobus: {(arac_full2["ARACTIPI"]==OTOBUS).sum():,}')
print(f'  Metrobus: {(arac_full2["ARACTIPI"]==METROBUS).sum():,}')

m4b = ols('ort_skor ~ C(GARAJ) + yas + log_sefer + ort_egim', data=arac_full2).fit()
m5 = ols('ort_skor ~ C(GARAJ) + yas + log_sefer + ort_egim + C(ARACTIPI)', data=arac_full2).fit()

print()
print('=== R2 KARSILASTIRMA ===')
print(f'M4 (yas+sefer+egim, ayni veri):  R2={m4b.rsquared:.4f}')
print(f'M5 (+ ARACTIPI):                 R2={m5.rsquared:.4f}')
print(f'Delta R2:                        {m5.rsquared - m4b.rsquared:+.4f}')

print()
print('=== ARACTIPI ETKISI (M5) ===')
for name, val in m5.params.items():
    if 'ARACTIPI' in name:
        p = m5.pvalues[name]
        print(f'  {name}: k={val:+.4f}, p={p:.6f}')

print()
print('=== KATSAYI KARSILASTIRMA: M4 vs M5 ===')
header = 'Garaj                          M4_k        M5_k        Degisim'
print(header)
print('-'*70)
hedef_garaj = ['Hasanpa\u015fa', 'Edirnekap\u0131', 'IKITELLIISLETTIRMEGARAJI2', 'Topkap\u0131', 'Anadolu', 'SULTANGAZIGARAJI']
for g in hedef_garaj:
    key_m4 = [k for k in m4b.params.keys() if g in k]
    key_m5 = [k for k in m5.params.keys() if g in k]
    if key_m4 and key_m5:
        k4 = m4b.params[key_m4[0]]
        k5 = m5.params[key_m5[0]]
        delta = k5 - k4
        if abs(k5) < abs(k4) - 0.05:
            yon = 'DUSTU (metrobus aciklayici)'
        elif abs(k5) > abs(k4) + 0.05:
            yon = 'ARTTI'
        else:
            yon = 'DEGISMEDI'
        print(f'{g:30s} {k4:+10.4f} {k5:+10.4f} {delta:+10.4f}  {yon}')

print()
print('=== YORUM ===')
delta_r2 = m5.rsquared - m4b.rsquared
if delta_r2 > 0.01:
    print(f'ARACTIPI eklemesi R2 {delta_r2:+.4f} arttirdi -> metrobus/otobus bagimsiz aciklayici')
else:
    print(f'ARACTIPI eklemesi R2 cok az artirdi ({delta_r2:+.4f}) -> garaj zaten metrobus etkisini tasiyor')
print('Hasanpasa/Edirnekapi katsayilari M5 te DUSTUYSE: kotuluk metrobus, garaj degil')
print('Dusmediyse: metrobus dahil bile bu garajlarda ayriksak sorun var')


ARACTIPI dagilimi (arac bazinda): {'Otobüs': 2728, 'Metrobüs': 774, 'Golf': 135, 'Yedek': 6, 'Kayıtsız Araç': 1}
M5 regresyon seti: 3,502 arac
  Otobus: 2,728
  Metrobus: 774

=== R2 KARSILASTIRMA ===
M4 (yas+sefer+egim, ayni veri):  R2=0.2574
M5 (+ ARACTIPI):                 R2=0.2585
Delta R2:                        +0.0011

=== ARACTIPI ETKISI (M5) ===
  C(ARACTIPI)[T.Otobüs]: k=+0.1840, p=0.023058

=== KATSAYI KARSILASTIRMA: M4 vs M5 ===
Garaj                          M4_k        M5_k        Degisim
----------------------------------------------------------------------
Hasanpaşa                         +0.1631    +0.3223    +0.1591  ARTTI
Edirnekapı                        +0.1418    +0.3022    +0.1604  ARTTI
IKITELLIISLETTIRMEGARAJI2         -0.2218    -0.2501    -0.0282  DEGISMEDI
Topkapı                           -1.5762    -1.6032    -0.0270  DEGISMEDI
SULTANGAZIGARAJI                  -0.1952    -0.2032    -0.0080  DEGISMEDI

=== YORUM ===
ARACTIPI eklemesi R2 cok az artirdi (+

---

## 17. Metrobus vs Otobus Ayri Karsilastirma (Saf Analiz)

Filonun kalpleri = otobus + metrobus. Karistirmak elma-armut karsilastirmasi.
Iki ayri evren olarak analiz:

1. Sadece Otobus: 10 garaj
2. Sadece Metrobus: 2-4 garaj (Edirnekapi, Hasanpasa + biraz Anadolu/SULTANGAZI)

Her evren icinde "saf" garaj kalite karsilastirmasi.


In [17]:
# BOLUM 17: Metrobus vs Otobus Ayri Karsilastirma
from scipy.stats import f_oneway, mannwhitneyu

print('=== SADECE OTOBUS EVRENI ===')
otobus = arac_full2[arac_full2['ARACTIPI']==OTOBUS].copy()
otobus_garaj = otobus.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_mean=('yas','mean'),
    ort_skor_mean=('ort_skor','mean'),
    cascade_mean=('cascade_oran','mean'),
).round(3).sort_values('ort_skor_mean')
print(f'Toplam {len(otobus):,} arac, {otobus["GARAJ"].nunique()} garaj')
print(otobus_garaj.to_string())

gruplar = [grp['ort_skor'].values for _, grp in otobus.groupby('GARAJ') if len(grp) >= 20]
if len(gruplar) >= 2:
    f_stat, p_f = f_oneway(*gruplar)
    print(f'\nOtobus evreninde garaj farki: F={f_stat:.2f}, p={p_f:.6e}')

print()
print('=== SADECE METROBUS EVRENI ===')
metrobus = arac_full2[arac_full2['ARACTIPI']==METROBUS].copy()
metrobus_garaj = metrobus.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_mean=('yas','mean'),
    ort_skor_mean=('ort_skor','mean'),
    cascade_mean=('cascade_oran','mean'),
).round(3).sort_values('ort_skor_mean')
print(f'Toplam {len(metrobus):,} arac, {metrobus["GARAJ"].nunique()} garaj')
print(metrobus_garaj.to_string())

gruplar_m = [grp['ort_skor'].values for _, grp in metrobus.groupby('GARAJ') if len(grp) >= 10]
if len(gruplar_m) >= 2:
    f_stat_m, p_f_m = f_oneway(*gruplar_m)
    print(f'\nMetrobus evreninde garaj farki: F={f_stat_m:.2f}, p={p_f_m:.6e}')

print()
print('=== OTOBUS vs METROBUS GENEL ===')
print(f'Otobus ort_skor:    {otobus["ort_skor"].mean():.3f}')
print(f'Metrobus ort_skor:  {metrobus["ort_skor"].mean():.3f}')
u_stat, p_u = mannwhitneyu(otobus['ort_skor'], metrobus['ort_skor'])
print(f'Mann-Whitney U: U={u_stat:.0f}, p={p_u:.6e}')
print(f'Otobus cascade:    {otobus["cascade_oran"].mean():.4f}')
print(f'Metrobus cascade:  {metrobus["cascade_oran"].mean():.4f}')

print()
print('=== EN IYI/KOTU GARAJ (saf evren icinde) ===')
print('Otobus en iyi 3:', otobus_garaj.head(3).index.tolist())
print('Otobus en kotu 3:', otobus_garaj.tail(3).index.tolist())
print('Metrobus en iyi:', metrobus_garaj.head(2).index.tolist())
print('Metrobus en kotu:', metrobus_garaj.tail(2).index.tolist())


=== SADECE OTOBUS EVRENI ===
Toplam 2,728 arac, 10 garaj
                           n_arac  yas_mean  ort_skor_mean  cascade_mean
GARAJ                                                                   
Topkapı                       151     1.119          2.072         0.058
Yunus                         229    12.000          3.359         0.089
SULTANGAZIGARAJI              375    12.160          3.444         0.128
IKITELLIGARAJI                423     9.248          3.473         0.084
IKITELLIISLETTIRMEGARAJI2     330    11.318          3.486         0.173
Anadolu                       317    17.852          3.559         0.109
Sarıgazi                      184    11.783          3.622         0.084
Kağıthane                     244    12.000          3.835         0.075
KURTKÖY                       346    12.066          3.853         0.140
Şahinkaya                     129    19.000          4.065         0.077

Otobus evreninde garaj farki: F=100.02, p=8.970326e-162

=== SADEC

---

## 18. Hasanpasa Cascade Alarm Hala Gecerli mi?

Bolum 3'te Hasanpasa cascade oran 0.201 = en kotu demistik (genel veride).
Ama Hasanpasa SAF METROBUS. Gercek karsilastirma: Hasanpasa metrobus vs Edirnekapi metrobus.

- Benzerlerse: Hasanpasa alarmi YANILTICI -> metrobus gercegi
- Hasanpasa belirgin yuksekse: alarm hala gecerli -> metrobus icinde bile sorun


In [18]:
# BOLUM 18: Hasanpasa vs Edirnekapi Metrobus Karsilastirma
mb = arac_full2[arac_full2['ARACTIPI']==METROBUS].copy()
mb_garaj = mb.groupby('GARAJ').agg(
    n_arac=('KAPINO','count'),
    yas_mean=('yas','mean'),
    ort_skor_mean=('ort_skor','mean'),
    cascade_mean=('cascade_oran','mean'),
    cascade_median=('cascade_oran','median'),
).round(4).sort_values('cascade_mean', ascending=False)

print('=== METROBUS EVRENI: GARAJ DETAY ===')
print(mb_garaj.to_string())

# Hasanpasa vs Edirnekapi
HP_KEY = 'Hasanpa\u015fa'
EK_KEY = 'Edirnekap\u0131'
hp = mb[mb['GARAJ']==HP_KEY]['cascade_oran'].values
ek = mb[mb['GARAJ']==EK_KEY]['cascade_oran'].values
print()
print(f'Hasanpasa metrobus cascade:    mean={hp.mean():.4f}, median={np.median(hp):.4f}, n={len(hp)}')
print(f'Edirnekapi metrobus cascade:   mean={ek.mean():.4f}, median={np.median(ek):.4f}, n={len(ek)}')

from scipy.stats import mannwhitneyu
u, p = mannwhitneyu(hp, ek, alternative='two-sided')
print(f'Mann-Whitney U: U={u:.0f}, p={p:.6e}')

np.random.seed(42)
diff_boot = np.array([
    np.random.choice(hp, len(hp), replace=True).mean() - np.random.choice(ek, len(ek), replace=True).mean()
    for _ in range(1000)
])
ci_low, ci_high = np.percentile(diff_boot, [2.5, 97.5])
print(f'Hasanpasa - Edirnekapi cascade farki CI: [{ci_low:.4f}, {ci_high:.4f}]')

print()
print('=== KARAR ===')
if p < 0.05 and (ci_low > 0 or ci_high < 0):
    if hp.mean() > ek.mean():
        pct = (hp.mean()/ek.mean()-1)*100
        print('Hasanpasa cascade Edirnekapi dan ANLAMLI YUKSEK -> ALARM GECERLI')
        print(f'  Metrobus icinde bile fark: +%{pct:.1f}')
        print('  Tamir kalitesi sorunu metrobus etkisinden bagimsiz var')
    else:
        print('Edirnekapi cascade Hasanpasa dan yuksek -> alarmi Edirnekapi ya cevir')
else:
    print('Hasanpasa ve Edirnekapi cascade BENZER (p>=0.05 veya CI 0 iceriyor)')
    print('  Hasanpasa alarmi YANILTICIYDI: bu sadece metrobus gercegi')
    print('  Bolum 3 yorumunu duzelt: "Hasanpasa kotu" yerine "metrobus yipratici"')

print()
print('=== ORT_SKOR KARSILASTIRMASI ===')
hp_s = mb[mb['GARAJ']==HP_KEY]['ort_skor'].values
ek_s = mb[mb['GARAJ']==EK_KEY]['ort_skor'].values
u2, p2 = mannwhitneyu(hp_s, ek_s, alternative='two-sided')
print(f'Hasanpasa ort_skor: {hp_s.mean():.3f}, Edirnekapi ort_skor: {ek_s.mean():.3f}, p={p2:.6e}')


=== METROBUS EVRENI: GARAJ DETAY ===
                  n_arac  yas_mean  ort_skor_mean  cascade_mean  cascade_median
GARAJ                                                                          
Hasanpaşa            319    8.0313         3.7632        0.1860          0.1935
Edirnekapı           381   11.8320         3.7508        0.1435          0.1364
SULTANGAZIGARAJI      38   13.0000         3.3690        0.1068          0.1043
Anadolu               36   13.0000         3.6295        0.0967          0.0833

Hasanpasa metrobus cascade:    mean=0.1860, median=0.1935, n=319
Edirnekapi metrobus cascade:   mean=0.1435, median=0.1364, n=381
Mann-Whitney U: U=77558, p=2.923293e-10
Hasanpasa - Edirnekapi cascade farki CI: [0.0291, 0.0559]

=== KARAR ===
Hasanpasa cascade Edirnekapi dan ANLAMLI YUKSEK -> ALARM GECERLI
  Metrobus icinde bile fark: +%29.6
  Tamir kalitesi sorunu metrobus etkisinden bagimsiz var

=== ORT_SKOR KARSILASTIRMASI ===
Hasanpasa ort_skor: 3.763, Edirnekapi ort_skor: